In [ ]:
# 175. Combine Two Tables

# Table: Person
# +-------------+---------+
# | Column Name | Type    |
# +-------------+---------+
# | personId    | int     |
# | lastName    | varchar |
# | firstName   | varchar |
# +-------------+---------+
# personId is the primary key (column with unique values) for this table.
# This table contains information about the ID of some persons and their first and last names.
 
# Table: Address
# +-------------+---------+
# | Column Name | Type    |
# +-------------+---------+
# | addressId   | int     |
# | personId    | int     |
# | city        | varchar |
# | state       | varchar |
# +-------------+---------+
# addressId is the primary key (column with unique values) for this table.
# Each row of this table contains information about the city and state of one person with ID = PersonId.
 
# Write a solution to report the first name, last name, city, and state of each person in the Person table. 
# If the address of a personId is not present in the Address table, report null instead.
# Return the result table in any order.
# Example 1:

# Input: 
# Person table:
# +----------+----------+-----------+
# | personId | lastName | firstName |
# +----------+----------+-----------+
# | 1        | Wang     | Allen     |
# | 2        | Alice    | Bob       |
# +----------+----------+-----------+
# Address table:
# +-----------+----------+---------------+------------+
# | addressId | personId | city          | state      |
# +-----------+----------+---------------+------------+
# | 1         | 2        | New York City | New York   |
# | 2         | 3        | Leetcode      | California |
# +-----------+----------+---------------+------------+
# Output: 
# +-----------+----------+---------------+----------+
# | firstName | lastName | city          | state    |
# +-----------+----------+---------------+----------+
# | Allen     | Wang     | Null          | Null     |
# | Bob       | Alice    | New York City | New York |
# +-----------+----------+---------------+----------+
# Explanation: 
# There is no address in the address table for the personId = 1 so we return null in their city and state.
# addressId = 1 contains information about the address of personId = 2.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Person table
person_data = [
    (1, "Wang", "Allen"),
    (2, "Alice", "Bob")
]

person_cols = ["personId", "lastName", "firstName"]

person_df = spark.createDataFrame(person_data, person_cols)


# Address table
address_data = [
    (1, 2, "New York City", "New York"),
    (2, 3, "Leetcode", "California")
]

address_cols = ["addressId", "personId", "city", "state"]

address_df = spark.createDataFrame(address_data, address_cols)


person_df.show()
address_df.show()

+--------+--------+---------+
|personId|lastName|firstName|
+--------+--------+---------+
|       1|    Wang|    Allen|
|       2|   Alice|      Bob|
+--------+--------+---------+

+---------+--------+-------------+----------+
|addressId|personId|         city|     state|
+---------+--------+-------------+----------+
|        1|       2|New York City|  New York|
|        2|       3|     Leetcode|California|
+---------+--------+-------------+----------+



In [10]:
import pyspark.sql.functions as F


In [15]:
df = person_df.join(address_df,"personId","left").select(F.col("firstName"),"lastName","city","state")
df.show()
df.explain(True)

+---------+--------+-------------+--------+
|firstName|lastName|         city|   state|
+---------+--------+-------------+--------+
|    Allen|    Wang|         NULL|    NULL|
|      Bob|   Alice|New York City|New York|
+---------+--------+-------------+--------+

== Parsed Logical Plan ==
'Project ['firstName, 'lastName, 'city, 'state]
+- Project [personId#0L, lastName#1, firstName#2, addressId#6L, city#8, state#9]
   +- Join LeftOuter, (personId#0L = personId#7L)
      :- LogicalRDD [personId#0L, lastName#1, firstName#2], false
      +- LogicalRDD [addressId#6L, personId#7L, city#8, state#9], false

== Analyzed Logical Plan ==
firstName: string, lastName: string, city: string, state: string
Project [firstName#2, lastName#1, city#8, state#9]
+- Project [personId#0L, lastName#1, firstName#2, addressId#6L, city#8, state#9]
   +- Join LeftOuter, (personId#0L = personId#7L)
      :- LogicalRDD [personId#0L, lastName#1, firstName#2], false
      +- LogicalRDD [addressId#6L, personId#7L, ci

In [17]:
# Select only required columns early
person_df_sel = person_df.select("personId", "firstName", "lastName")
address_df_sel = address_df.select("personId", "city", "state")

# Broadcast join (assuming address is small)
df = person_df_sel.alias("p") \
    .join(
        F.broadcast(address_df_sel).alias("a"),
        on="personId",
        how="left"
    ) \
    .select(
        F.col("p.firstName"),
        F.col("p.lastName"),
        F.col("a.city"),
        F.col("a.state")
    )

df.show()

+---------+--------+-------------+--------+
|firstName|lastName|         city|   state|
+---------+--------+-------------+--------+
|    Allen|    Wang|         NULL|    NULL|
|      Bob|   Alice|New York City|New York|
+---------+--------+-------------+--------+



In [18]:
df.explain(True)

== Parsed Logical Plan ==
'Project ['p.firstName, 'p.lastName, 'a.city, 'a.state]
+- Project [personId#0L, firstName#2, lastName#1, city#8, state#9]
   +- Join LeftOuter, (personId#0L = personId#7L)
      :- SubqueryAlias p
      :  +- Project [personId#0L, firstName#2, lastName#1]
      :     +- LogicalRDD [personId#0L, lastName#1, firstName#2], false
      +- SubqueryAlias a
         +- ResolvedHint (strategy=broadcast)
            +- Project [personId#7L, city#8, state#9]
               +- LogicalRDD [addressId#6L, personId#7L, city#8, state#9], false

== Analyzed Logical Plan ==
firstName: string, lastName: string, city: string, state: string
Project [firstName#2, lastName#1, city#8, state#9]
+- Project [personId#0L, firstName#2, lastName#1, city#8, state#9]
   +- Join LeftOuter, (personId#0L = personId#7L)
      :- SubqueryAlias p
      :  +- Project [personId#0L, firstName#2, lastName#1]
      :     +- LogicalRDD [personId#0L, lastName#1, firstName#2], false
      +- SubqueryAlia